# Phase 1 SFT — Qwen 2.5-0.5B on GSM8K (Colab T4)

End-to-end Supervised Fine-Tuning run for Phase 1: train Qwen 2.5-0.5B on GSM8K math word problems using the same packing pipeline tested locally with tiny-gpt2.

**What changes from the local script:**
- Model: `sshleifer/tiny-gpt2` (~1MB) → `Qwen/Qwen2.5-0.5B` (~1GB)
- Device: CPU → CUDA T4
- Dtype: float32 → bfloat16 (halves memory, T4 supports it natively)
- Batch size: 2 → 4 (T4 has 15GB VRAM; 4 fits comfortably)
- Steps: 10 → configurable (start with 50–100 to see real learning)

**Before running:** make sure you set the runtime to T4 GPU.
Runtime → Change runtime type → T4 GPU.

---

## Step 1 — verify the GPU

If `nvidia-smi` errors or shows no GPU, the runtime is not set to T4. Stop here and switch it. Training Qwen 0.5B on a CPU runtime would take hours; on T4 it takes seconds per step.

In [ ]:
!nvidia-smi

: 

## Step 2 — clone the repo and install dependencies

We clone the public finpost repo and install it in editable mode (`pip install -e .`). Editable mode means we install the *source tree* itself as the package — so `from finpost.training.dataset import make_loaders` works, exactly as it does on your local machine. No copy-pasting code into cells.

The `transformers`, `torch`, `datasets`, `pyyaml`, `pydantic` deps are listed in pyproject.toml and pip will resolve them. Colab usually has compatible versions of torch and transformers preinstalled, so this finishes in 1–2 minutes.

In [ ]:
!git clone https://github.com/shannan-liu1/finpost.git
%cd finpost
!pip install -q -e .

## Step 3 — imports and config

We import the same modules the local script uses. The only difference is the Config we build below: pointed at Qwen 2.5-0.5B with bfloat16 weights and a batch size tuned for T4.

In [ ]:
from __future__ import annotations

from typing import Any

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from finpost.training.config import (
    Config,
    DataConfig,
    ModelConfig,
    PackingConfig,
    TrainingConfig,
)
from finpost.training.dataset import make_loaders
from finpost.training.sft import compute_masked_ce_loss

# Confirm the Python and PyTorch view of the GPU agrees with nvidia-smi.
# torch.cuda.is_available() should be True; the device name should
# include 'T4' (or 'A100' / 'V100' on a paid runtime).
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:        ', torch.cuda.get_device_name(0))
    print('VRAM (GB):     ', torch.cuda.get_device_properties(0).total_memory / 1e9)

In [ ]:
# Phase 1 base model. 0.49B params per the model card; bf16 weights
# are about 1 GB on disk and in VRAM. Same HuggingFace identifier
# loads both the tokenizer and the model.
MODEL_ID = 'Qwen/Qwen2.5-0.5B'

# Steps to run. Each T4 step is ~1-3 seconds depending on batch and
# sequence length, so 50 steps ~= 1-3 minutes wall-clock. For a real
# Phase 1 run you would set this to several thousand and let it cook.
NUM_STEPS = 50

# Packed rows per step. T4 has 15GB VRAM:
#   model bf16 weights:   ~1.0 GB
#   AdamW state (fp32):   ~4.0 GB  (two momentum buffers, full precision)
#   gradients:            ~1.0 GB
#   activations (bs=4):   ~3-5 GB depending on seq_len
# Total ~10-12 GB, leaving 3-5 GB headroom. Increase to 8 if you want.
BATCH_SIZE = 4

# Token cap per packed row. GSM8K combined p95 is 312 tokens, so
# 1024 fits a few documents per row. Keep modest on a free T4 to
# avoid OOM at backward time.
MAX_SEQ_LEN = 1024

config = Config(
    model=ModelConfig(
        base_model_id=MODEL_ID,
        # bfloat16 has the same dynamic range as float32 (8-bit
        # exponent) but only 7-bit mantissa - fine for forward and
        # backward passes in transformer training. Halves the memory
        # vs fp32 and runs natively on T4 tensor cores.
        dtype='bfloat16',
        # Phase 1 model ships safetensors. Always prefer it over
        # pickle (.bin) weights for any model you do not control.
        use_safetensors=True,
    ),
    data=DataConfig(
        sources=['gsm8k'],
        val_split_pct=5.0,
        seed=42,
    ),
    training=TrainingConfig(
        # Required by the schema; the validator only enforces
        # warmup_steps < max_steps. Our actual loop length is
        # NUM_STEPS above.
        max_steps=10_000,
        warmup_steps=10,
        # 1e-4 is a safe SFT learning rate for a 0.5B base model.
        # Lower (5e-5) is more conservative; higher (3e-4) trains
        # faster but is more likely to spike. Stay at 1e-4 for the
        # first run.
        lr=1e-4,
        per_device_batch_size=BATCH_SIZE,
    ),
    packing=PackingConfig(
        max_seq_len=MAX_SEQ_LEN,
        # Cross-document attention isolation. The collator builds a 4D
        # attention mask that blocks attention across packed-document
        # boundaries AND enforces the causal triangle within each
        # document. The training cell below passes that 4D mask plus
        # the per-document position_ids to the model.
        isolate_documents=True,
    ),
)

print('Config built:')
print(f'  model       {config.model.base_model_id}')
print(f'  dtype       {config.model.dtype}')
print(f'  batch       {config.training.per_device_batch_size}')
print(f'  max_seq_len {config.packing.max_seq_len}')
print(f'  lr          {config.training.lr}')

## Step 4 — load tokenizer and model

This is the slow cell — Qwen 2.5-0.5B safetensors weights download from the HuggingFace Hub on first run (~1 GB) and load into GPU memory. After the first run, the file is cached at `~/.cache/huggingface/hub/` and subsequent loads take seconds.

**`tokenizer.pad_token` fallback**: Qwen's tokenizer does have a pad token, so the `if` branch should not fire here. We keep the same defensive code as the local script for parity. The reason this matters: the collator pads packed rows to a uniform width using `tokenizer.pad_token_id`; if that returned `None` we'd silently use 0, which on Qwen happens to be a real token (`!`).

In [ ]:
device = torch.device('cuda')

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(config.model.base_model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f'  pad_token_id = {tokenizer.pad_token_id}')
print(f'  eos_token_id = {tokenizer.eos_token_id}')

print('\nLoading model (downloads ~1GB on first run)...')
dtype = getattr(torch, config.model.dtype)
model = AutoModelForCausalLM.from_pretrained(
    config.model.base_model_id,
    dtype=dtype,
    use_safetensors=config.model.use_safetensors,
).to(device)
model.train()

# Sanity check VRAM usage after the model is on GPU. This number
# should be roughly the bf16 parameter size (~1 GB for Qwen 0.5B);
# if it is much higher, dtype defaulted to fp32 somewhere.
alloc_gb = torch.cuda.memory_allocated() / 1e9
print(f'\nGPU memory allocated after model load: {alloc_gb:.2f} GB')

## Step 5 — build the data loaders

Same call as the local script. `make_loaders` runs the full data pipeline:

1. Loads GSM8K from the HF datasets cache (downloads on first run).
2. Wraps in `PhasedSFTDataset` for train and val splits, deterministic stratified split per `config.data.seed`.
3. Wraps both in `DataLoader` with `PackingCollator` as the collate function. Each batch the loader yields is a dict of `{input_ids, labels, attention_mask, position_ids, document_boundaries}`.

In [ ]:
print('Building data loaders...')
train_loader, val_loader = make_loaders(config, tokenizer)
print(f'Train examples (post-split): {len(train_loader.dataset)}')
print(f'Val   examples (post-split): {len(val_loader.dataset)}')

# Peek at one batch to confirm the collator is producing the right
# shapes before we start training.
peek = next(iter(train_loader))
print(f"\nFirst batch shapes:")
print(f"  input_ids:      {tuple(peek['input_ids'].shape)}")
print(f"  labels:         {tuple(peek['labels'].shape)}")
print(f"  attention_mask: {tuple(peek['attention_mask'].shape)}")
print(f"  response tokens (non-IGNORE): {(peek['labels'] != -100).sum().item()}")

## Step 6 — training loop

The training loop body is identical to `train_packed_step` in `scripts/sft_local_train.py`. We inline it here so all the GPU-relevant logic is visible in this notebook (no jumping between cell and source file mid-run).

Per step:
1. Move tensors to GPU. Cast `attention_mask` to `bool` because PyTorch's SDPA attention requires bool or float, not int64.
2. Zero gradients from the prior step.
3. Forward pass with `input_ids`, the 4D `attention_mask` (document isolation + causal), and per-document `position_ids`.
4. Compute masked cross-entropy loss (prompt and padding positions excluded).
5. Backprop.
6. AdamW step.
7. Log loss and packing geometry.

**Watch the loss**: a healthy run drops from ~3-4 toward ~1-2 over 50 steps on GSM8K. Qwen 2.5-0.5B is a base model already exposed to math during pretraining, so SFT loss starts much lower than the random tiny-gpt2 ~10.8.

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.training.lr,
    weight_decay=config.training.weight_decay,
)

losses: list[float] = []
loader_iter = iter(train_loader)

print(f'Running {NUM_STEPS} steps:\n')
for step in range(NUM_STEPS):
    try:
        batch = next(loader_iter)
    except StopIteration:
        # New epoch.
        loader_iter = iter(train_loader)
        batch = next(loader_iter)

    # --- one training step ---
    input_ids = batch['input_ids'].to(device)
    labels = batch['labels'].to(device)
    # Cast to bool: SDPA requires bool or float, not int64.
    # True = "attend here", False = "block".
    attention_mask = batch['attention_mask'].to(device).bool()
    position_ids = batch['position_ids'].to(device)

    optimizer.zero_grad()
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        position_ids=position_ids,
    )
    loss = compute_masked_ce_loss(outputs.logits, labels)
    loss.backward()
    optimizer.step()
    # ---

    losses.append(loss.item())
    rows, seq_len = batch['input_ids'].shape
    response_tokens = (batch['labels'] != -100).sum().item()
    print(
        f'  step {step:3d}: loss={loss.item():.4f}  '
        f'shape=({rows}x{seq_len})  resp_tokens={response_tokens}'
    )

print()
if any(not torch.isfinite(torch.tensor(v)).item() for v in losses):
    print(f'FAIL: NaN or inf in loss curve')
elif losses[-1] < losses[0]:
    drop = losses[0] - losses[-1]
    print(f'OK: loss {losses[0]:.4f} -> {losses[-1]:.4f} (drop {drop:.4f})')
else:
    print(f'WARN: loss did not decrease; try more steps or higher --lr')

## Step 7 — plot the loss curve

Quick visual check. A healthy SFT curve:
- monotonically decreasing on average (with some noise),
- no spikes to inf or NaN,
- visible plateau forming as the model fits the dataset.

If the curve is flat, learning rate may be too low. If it spikes, learning rate is too high or there's a numerical-stability issue (try fp32 or lower lr).

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(losses, marker='o', markersize=3, linewidth=1)
plt.xlabel('step')
plt.ylabel('loss')
plt.title(f'{config.model.base_model_id} on GSM8K — first {len(losses)} steps')
plt.grid(alpha=0.3)
plt.show()

## Next steps

If this run looks healthy:

1. **Longer run** — bump `NUM_STEPS` to 500 or 1000 and re-run. GSM8K is small (~7k examples); a few epochs is reasonable.
2. **Validation** — call `next(iter(val_loader))` and compute loss without `optimizer.step()` to track val loss alongside train loss.
3. **Save the checkpoint** — `model.save_pretrained('./checkpoint')` and download via the Files panel, or push to HF Hub if you have a token.
4. **Wire in the real trainer loop** — when issues 03-06 of `phase1-sft-trainer` land, swap this hand-written loop for the production trainer with warmup+cosine LR, gradient accumulation, gradient clipping, and periodic validation.

If the run looks unhealthy (spikes, NaN, flat loss): drop `lr` to 5e-5, drop `BATCH_SIZE` to 2, switch dtype to `'float32'` to rule out a bf16 numerical issue, and re-run.